In [1]:
from datasets import Dataset
from knowledge_utils import DocProcessor
import pandas as pd
import json
import os

os.environ["SDG_LANG"] = "Portuguese"
os.environ["SDG_LANG_CODE"] = "pt-br"

from sdg_hub import Flow, FlowRegistry

# Required to run the flow with async mode
import nest_asyncio

nest_asyncio.apply()

# Step 1: Document Processing Pipeline
# Define the directory containing raw documents to be processed
data_dir = "document_collection/"
user_config_path = f"{data_dir}/qna.yaml"

# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
!python ../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../docling_v2_config.yaml

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'knowledge_utils'

In [ ]:
# Step 3: Load Processed Document
import glob

# In our example above docling step produces markdown of all the pdf files in the document_collection
with open(glob.glob(f"{data_dir}/*.md")[0], "r") as f:
    text = f.read()

In [ ]:
# Step 4: Text Chunking and Dataset Creation

from markdown_it import MarkdownIt
from typing import List
import datasets


def chunk_markdown(text: str, max_tokens: int = 200, overlap: int = 50) -> List[str]:
    """
    Splits Markdown text into chunks at block-level elements
    (headings, paragraphs, lists, tables, code, blockquotes).
    Adds overlap (in words) between all consecutive chunks.

    Args:
        text: The markdown text to be chunked
        max_tokens: Maximum number of words per chunk
        overlap: Number of overlapping words between consecutive chunks

    Returns:
        List of text chunks with specified overlap
    """

    # Initialize markdown parser to understand document structure
    md = MarkdownIt()
    tokens = md.parse(text)

    # Group tokens into block-level segments to preserve markdown structure
    # This ensures we don't split in the middle of headings, lists, etc.
    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    # Split blocks into chunks with overlap to maintain context continuity
    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                # Emit a complete chunk
                chunks.append(" ".join(current_words))
                # Prepare next buffer with overlap from the end of this chunk
                # This ensures context continuity between chunks
                current_words = current_words[-overlap:] if overlap > 0 else []

    # Add any remaining words as the final chunk
    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


chunks = chunk_markdown(text, max_tokens=5000, overlap=1000)
#print(chunks)

In [ ]:
# Create dataset
dataset = Dataset.from_dict({
    "document": chunks,
    "document_outline": ["Feriados e Emendas de 2026"] * len(chunks)
})

input_dataset = dataset

In [ ]:
# Get the RAG Evaluation flow
flow_name = "RAG Evaluation Dataset Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)

flow = Flow.from_yaml(flow_path)

In [ ]:
def set_model_config(flow_object):
    """Configure the model for the flow based on environment variables."""
    model = os.getenv("INFERENCE_MODEL", "RedHatAI/gpt-oss-20b")
    api_base = os.getenv("URL", "http://granite-ai-inference-server-ocp.apps.cluster-jzfpx.jzfpx.sandbox2518.opentlc.com/v1")
    api_key = os.getenv("API_KEY", "not")
    
    if model and not model.startswith("openai/") and not model.startswith("ollama/"):
        model = "openai/" + model
    
    if not model:
        raise ValueError("INFERENCE_MODEL environment variable must be set")
    
    print(f"Configuring model: {model}")
    
    flow_object.set_model_config(
        model=model,
        api_base=api_base if api_base else None,
        api_key=api_key if api_key else None,
    )
    
    return flow_object

# Configure the model
flow = set_model_config(flow)

In [ ]:
# Get runtime parameters
max_concurrency = int(os.getenv("MAX_CONCURRENCY", "10"))

# Optional: Configure runtime parameters for specific blocks
runtime_params = {}

print("This may take several minutes depending on dataset size and model speed...\n")

# Generate the dataset
generated_data = flow.generate(
    input_dataset, 
    runtime_params=runtime_params, 
    max_concurrency=max_concurrency
)

In [ ]:
df = generated_data.to_pandas()

print(f"Total records: {len(df)}")
print("\nColumns:", list(df.columns))

print("\nSAMPLE GENERATED RECORDS")

sample_cols = ["topic", "question", "response", "ground_truth_context"]

for i, row in df.head(3).iterrows():
    print("\n")
    for col in sample_cols:
        if col in df:
            val = row[col]
            text = str(val)
            #if len(text) > 200:
            #    text = text[:200] + "..."
            print(f"{col.title()}: {text}")

In [ ]:
# Initialize processor
dp = DocProcessor(data_dir, user_config_path=user_config_path)

# Process markdown files
list_md_files = [f"{data_dir}/Feriados e Emendas de 2026.md"]
seed_data = dp.get_processed_markdown_dataset(list_md_files)

# Save seed data
output_dir = "sdg_demo_output"
seed_data_path = f"{output_dir}/seed_data.jsonl"
seed_data.to_json(seed_data_path, orient="records", lines=True, force_ascii=False)